# Chapter 10 &mdash; Brzozowski's Insight: an RE as its Own State

**Concept 1 of the Chapter 10 decomposition:** *Brzozowski's Insight: an RE as its own State, Morphing as it Eats*

A DFA changes state; an RE has no state, so it must rewrite itself into a new RE as it eats input.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter10/Concept-RE-As-Its-Own-State/Concept-RE-As-Its-Own-State.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.Def_RE2NFA     import *
from jove.Def_rederiv    import *

import jove; print('Jove loaded from', list(jove.__path__)[0])

## 1. The idea


A DFA matches by **changing state**. A regular expression has no states &mdash; so how can
it match without first being compiled into a machine?

**Brzozowski's answer:** let the expression **morph into a new expression** as it eats
each symbol. The *derivative* $E_c$ is "what is left of $E$ after consuming $c$":

$$L(E_c) = \{w : cw \in L(E)\}.$$

The expression **is** the state. No NFA, no subset construction, no table &mdash; just a
rewrite per symbol, and a final "does this expression accept $\varepsilon$?" test.

## 2. Definitions

### The matcher

In [ ]:
# --- the derivative matcher, in full -------------------------------------
# AST forms produced by re2ast:
#    ('@','@')            epsilon
#    ('str', c)           a single symbol
#    ('+', (E1, E2))      union
#    ('.', (E1, E2))      concatenation
#    ('*', E)             star
#    ('!', E)             negation
#    ('&', (E1, E2))      intersection
EPS   = ('@', '@')
PHI   = ('phi', 'phi')          # the empty language -- not produced by the
                                # parser, but the derivative rules need it

def nullable(E):
    t = E[0]
    if t == '@'  : return True
    if t == 'phi': return False
    if t == 'str': return False
    if t == '+'  : return nullable(E[1][0]) or  nullable(E[1][1])
    if t == '&'  : return nullable(E[1][0]) and nullable(E[1][1])
    if t == '.'  : return nullable(E[1][0]) and nullable(E[1][1])
    if t == '*'  : return True
    if t == '!'  : return not nullable(E[1])
    raise ValueError(E)

def dv(c, E):
    t = E[0]
    if t == '@'  : return PHI
    if t == 'phi': return PHI
    if t == 'str': return EPS if E[1] == c else PHI
    if t == '+'  : return ('+', (dv(c, E[1][0]), dv(c, E[1][1])))
    if t == '&'  : return ('&', (dv(c, E[1][0]), dv(c, E[1][1])))
    if t == '*'  : return ('.', (dv(c, E[1]), E))
    if t == '!'  : return ('!', dv(c, E[1]))
    if t == '.'  :
        E1, E2 = E[1]
        left = ('.', (dv(c, E1), E2))
        return ('+', (left, dv(c, E2))) if nullable(E1) else left
    raise ValueError(E)

def matches(s, E):
    for ch in s:
        E = dv(ch, E)
    return nullable(E)

def rmatch(restr, s):
    return matches(s, re2ast(restr)[0])

### Watch an expression morph

In [ ]:
def trace(restr, s):
    E = re2ast(restr)[0]
    print("start        : %s" % (E,))
    for ch in s:
        E = dv(ch, E)
        print("after '%s'    : nullable=%-6s" % (ch, nullable(E)))
    return nullable(E)

<!-- nav-strip -->

---

&larr;&nbsp;[Ch9&nbsp;8.&nbsp;Closure Results for Regular Languages](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter9/Concept-Closure-Results/Concept-Closure-Results.ipynb) &nbsp;&middot;&nbsp; [**Chapter 10** index](https://github.com/ganeshutah/Jove/blob/master/Chapter10/README.md) &nbsp;&middot;&nbsp; [Ch10&nbsp;2.&nbsp;Why Derivatives Matter: Negation and Intersection Without Determinization](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter10/Concept-Negation-And-Intersection/Concept-Negation-And-Intersection.ipynb)&nbsp;&rarr;

---

## 3. Tests

The expression rewrites itself, symbol by symbol.

In [ ]:
print("matching '001' against 0*1")
print("result :", trace("0*1", "001"))
assert rmatch("0*1", "001")

The derivative definition, checked directly: $L(E_c) = \{w : cw\in L(E)\}$.

In [ ]:
from itertools import product
def lang_of(E, n=6):
    return {''.join(p) for k in range(n+1) for p in product('01', repeat=k)
            if matches(''.join(p), E)}

E = re2ast("(0+1)*01")[0]
for ch in '01':
    lhs = lang_of(dv(ch, E), 5)
    rhs = {w for w in lang_of(E, 6) if w.startswith(ch)}
    rhs = {w[1:] for w in rhs}
    print("c=%s : L(E_c) == {w : cw in L(E)} ? %s" % (ch, lhs == rhs))
    assert lhs == rhs

No machine was built &mdash; compare with the DFA route.

In [ ]:
D = min_dfa(nfa2dfa(re2nfa("(0+1)*01")))
strs = [''.join(p) for k in range(10) for p in product('01', repeat=k)]
assert all(rmatch("(0+1)*01", s) == accepts_dfa(D, s) for s in strs)
print("derivative matcher agrees with the %d-state minimal DFA on all %d strings"
      % (len(D["Q"]), len(strs)))
print("\n...but the matcher never constructed a state, a table, or a subset.")

## 4. Exercises


1. Compute $(0^*1)_0$ by hand and compare with `dv('0', re2ast("0*1")[0])`.
2. What is the derivative of $E$ with respect to a symbol not in $L(E)$'s alphabet?
3. Why is "the expression is the state" more than a slogan? (Concept 8.)

In [ ]:
# Your work for the exercises above.

## 5. Where next

In [ ]:
# Previous / next, and a search box for all 252 concepts.
# Type a chapter (Chapter7, ch7) or words from a title (pumping, subset).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:  load_here('Chapter7/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter10/Concept-RE-As-Its-Own-State')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')